In [58]:
import brevitas.nn as qnn
from models import smaller
from utils import get_model_cfg, data_binarization, data_preprocess
from sklearn.model_selection import train_test_split 
import torch
import pandas as pd

WEIGHT_PATH = '/home/sgeraci/slu/inet-hynn/pysrc/models/best_fold_distilled_vanilla_acc0.905.pth'
TRAIN_DATASET_PATH = '/home/sgeraci/slu/inet-hynn/datasets/UNSW_NB15_training-set.csv'
EVALU_DATASET_PATH = '/home/sgeraci/slu/inet-hynn/datasets/UNSW_NB15_testing-set.csv'
CONF_PATH = '/home/sgeraci/slu/inet-hynn/pysrc/cfgs/default'
selected_feats = ['sttl', 'ct_srv_dst', 'ct_dst_src_ltm', 'ct_srv_src', 'sbytes', 'smean', 'synack', 'dmean', 'tcprtt', 'ct_src_ltm', 'dbytes', 'service', 'ct_dst_sport_ltm', 'dloss', 'dload', 'ct_dst_ltm', 'sloss', 'swin', 'label']

In [59]:
data_tr = pd.read_csv(TRAIN_DATASET_PATH, delimiter=',')
data_te = pd.read_csv(EVALU_DATASET_PATH, delimiter=',')
dict = {
    'categorical_features_values': 6,
    'continuous_features_values': 50,
    'list_drop': [
        'id',
        'attack_cat'
    ]
}
x_tmp = data_te[data_te.columns[-1]]
y_tmp = data_te[data_te.columns[:-1]]
x_tmp_tr, x_tmp_te, y_tmp_tr, y_tmp_te = train_test_split(x_tmp, y_tmp, test_size=0.3, random_state=42)
tmp     = pd.concat([y_tmp_tr, x_tmp_tr], axis=1)
data_te = pd.concat([y_tmp_te, x_tmp_te], axis=1)
data_tr = pd.concat([data_tr, tmp], ignore_index=True)
data_tr = data_tr[selected_feats]
data_te = data_te[selected_feats]
X_tr_df, Y_tr, _ = data_preprocess(data_tr, dict, binarization=True)
X_te_df, Y_te, _ = data_preprocess(data_te, dict, binarization=True)
train_idx = X_tr_df.shape[0]
X_tmp = pd.concat([X_tr_df, X_te_df], ignore_index=True)

_, nn_size = data_binarization(samples=X_tmp.astype('int'), get_only_size=True)

Feature count before one-hot encoding: 18
Feature count after one-hot encoding: 23
Added   features: ['service_0', 'service_1', 'service_2', 'service_3', 'service_4', 'service_5']
Removed features: ['service']
Feature count before one-hot encoding: 18
Feature count after one-hot encoding: 23
Added   features: ['service_0', 'service_1', 'service_2', 'service_3', 'service_4', 'service_5']
Removed features: ['service']
Feat: sttl Bit width: 8 | Max value 255
Feat: ct_srv_dst Bit width: 6 | Max value 36
Feat: ct_dst_src_ltm Bit width: 6 | Max value 36
Feat: ct_srv_src Bit width: 6 | Max value 36
Feat: sbytes Bit width: 4 | Max value 9
Feat: smean Bit width: 3 | Max value 6
Feat: synack Bit width: 1 | Max value 1
Feat: dmean Bit width: 3 | Max value 6
Feat: tcprtt Bit width: 1 | Max value 1
Feat: ct_src_ltm Bit width: 5 | Max value 25
Feat: dbytes Bit width: 4 | Max value 10
Feat: ct_dst_sport_ltm Bit width: 5 | Max value 18
Feat: dloss Bit width: 5 | Max value 24
Feat: dload Bit width: 4 |

In [60]:
bin_w = torch.load(WEIGHT_PATH)
cfg = get_model_cfg()
model = smaller(cfg, nn_size)
model.load_state_dict(torch.load(WEIGHT_PATH, map_location="cpu"))
model.eval()

SmallerNN(
  (features): ModuleList(
    (0): QuantIdentity(
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (act_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (fused_activation_quant_proxy): FusedActivationQuantProxy(
          (activation_impl): Identity()
          (tensor_quant): ClampedBinaryQuant(
            (scaling_impl): ConstScaling(
              (restrict_clamp_scaling): _RestrictClampValue(
                (clamp_min_ste): Identity()
                (restrict_value_impl): FloatRestrictValue()
              )
              (restrict_init_module): Identity()
              (value): StatelessBuffer()
            )
            (bit_width): BitWidthConst(
              (bit_width): StatelessBuffer()
            )
            (zero_point): StatelessBuffer()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
            (tensor_clam

In [61]:
binary_weights = []
for name, module in model.named_modules():
    if isinstance(module, qnn.QuantLinear):
        qt: torch.Tensor = module.weight_quant(module.weight)
        bw = qt.int().detach().cpu().numpy()
        print(f"{name}.weight → binary shape ({bw.shape[1]},{bw.shape[0]})")
        binary_weights.append((name, bw.tolist()))

str_bin_w = ""

for l, l_w in binary_weights:
    for w in l_w:
        print(w)

    




features.1.weight → binary shape (84,128)
features.4.weight → binary shape (128,64)
features.7.weight → binary shape (64,32)
features.10.weight → binary shape (32,2)
[-1, -1, 1, 1, -1, -1, -1, 1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, 1, -1, 1, 1, -1, 1, 1, 1, 1, -1, -1, -1, 1, -1, 1, -1, 1, -1, -1, 1, -1, 1, 1, -1, 1, 1, -1, -1, 1, 1, -1, 1, 1, 1, 1, -1, -1, -1, 1, 1, 1, -1, -1, -1, -1, 1, 1, -1, -1, 1, 1, -1, -1, 1, 1, 1, 1, 1, -1, -1, 1, -1, 1, 1, -1, 1]
[-1, 1, -1, -1, -1, -1, 1, 1, 1, -1, 1, 1, 1, -1, 1, 1, 1, 1, -1, 1, -1, 1, -1, -1, -1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, -1, -1, -1, 1, -1, -1, 1, -1, -1, -1, -1, -1, 1, -1, -1, -1, 1, -1, 1, -1, 1, -1, 1, 1, 1, 1, -1, 1, -1, -1, 1, 1, -1, 1, 1, -1, -1, 1, 1, -1, -1, 1, 1, 1, 1, -1, 1, -1]
[-1, -1, -1, -1, -1, -1, -1, 1, 1, 1, 1, -1, -1, -1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 1, -1, 1, 1, 1, 1, -1, -1, -1, -1, 1, 1, 1, -1, -1, -1, -1, -1, 1, -1, 1, -1, 1, 1, 1, -1, -1, 1, 1, -1, 1, 1, 1, -1, -1, -1, 1, 1,